In [ ]:
import pandas as pd
import seaborn as sns
import altair as alt
import plotly.express as px

from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import r2_score


# I took this dataset from this link "https://www.kaggle.com/datasets/seanlim123/world-happiness-cleaned" and this dataset provides information about the level of happiness from various factors that will be analyzed here.

In [ ]:
df = pd.read_csv('/content/sample_data/2022_cleaned.csv')
df.head(5)

,Country,Region,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption),Generosity,Dystopia Residual
0,Finland,Western Europe,1,7.8210,1.891628,1.258108,0.775206,0.735590,0.533658,0.108733,2.518052
1,Denmark,Western Europe,2,7.6362,1.952595,1.242681,0.776644,0.718918,0.532079,0.187626,2.225632
2,Iceland,Western Europe,3,7.5575,1.935726,1.319914,0.802622,0.718194,0.191204,0.269616,2.320185
3,Switzerland,Western Europe,4,7.5116,2.025970,1.226074,0.822048,0.676947,0.461004,0.146822,2.152746
4,Netherlands,Western Europe,5,7.4149,1.944578,1.205848,0.786738,0.650682,0.419083,0.271076,2.136937


# ABSTRACT / LATAR BELAKANG :
Dataset Background:

The World Happiness Report is an annual report compiled by the Sustainable Development Solutions Network (SDSN) that measures the level of happiness in various countries. The assessment is based on socio-economic indicators such as GDP per capita, social support, healthy life expectancy, freedom to make life decisions, and levels of corruption.

Dataset Purpose:
This dataset aims to:

1. measure the level of happiness in a country,
2. analyze socio-economic factors that influence well-being,
3. provide policy recommendations to improve the quality of life for the community.

# PERFORMING DATA EXPLORATION AND PREPROCCESSING

In [ ]:
df.info()
# Here are the results of an initial inspection of the dataset

# structure using df.info():

# The dataset has 146 rows and 11 columns.

# The Country and Region columns are object types,
# so they are categorized as categorical data.

# There are many numeric columns (Happiness Score, Economy,
# Family, Health, Freedom, Trust, Generosity, etc.).

# There are no missing values, indicated by a Non-Null Count equal
# to the number of rows (146).

# The numeric data type is predominantly float64, which is
# suitable for statistical analysis.

# Initial conclusion:
# The dataset is clean, has a combination of categorical and numeric data, and
# is suitable for further statistical analysis and exploration.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 146 entries, 0 to 145
Data columns (total 11 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Country                        146 non-null    object 
 1   Region                         146 non-null    object 
 2   Happiness Rank                 146 non-null    int64  
 3   Happiness Score                146 non-null    float64
 4   Economy (GDP per Capita)       146 non-null    float64
 5   Family                         146 non-null    float64
 6   Health (Life Expectancy)       146 non-null    float64
 7   Freedom                        146 non-null    float64
 8   Trust (Government Corruption)  146 non-null    float64
 9   Generosity                     146 non-null    float64
 10  Dystopia Residual              146 non-null    float64
dtypes: float64(8), int64(1), object(2)
memory usage: 12.7+ KB


# Check Data Null


In [ ]:
df.isna().sum()

,0
Country,0
Region,0
Happiness Rank,0
Happiness Score,0
Economy (GDP per Capita),0
Family,0
Health (Life Expectancy),0
Freedom,0
Trust (Government Corruption),0
Generosity,0


# Check Data Duplicated

In [ ]:
df.duplicated().sum()

np.int64(0)

# Best Practice DropNa And Duplicated

In [ ]:
df.dropna(inplace=True) # artinya: ubah data di tempat, langsung di dataframe asli.
# Tanpa membuat salinan baru TANPA HARUS BUAT ASSIGN df = df Tanpa INplace = True
df.drop_duplicates(inplace=True)

In [ ]:
df.describe()

,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption),Generosity,Dystopia Residual
count,146.000000,146.000000,146.000000,146.000000,146.000000,146.000000,146.000000,146.000000,146.000000
mean,73.500000,5.553575,1.410429,0.905834,0.586146,0.517225,0.154758,0.147372,1.831807
std,42.290661,1.086814,0.421655,0.280123,0.176352,0.145833,0.127522,0.082763,0.534971
min,1.000000,2.403800,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.187416
25%,37.250000,4.888550,1.095578,0.732114,0.462953,0.440506,0.068087,0.088723,1.555475
50%,73.500000,5.568700,1.445134,0.957799,0.621445,0.543633,0.119459,0.132417,1.894558
75%,109.750000,6.305025,1.785087,1.114658,0.720065,0.625763,0.198213,0.197554,2.152897
max,146.000000,7.821000,2.209395,1.319914,0.941829,0.740039,0.586828,0.467926,2.844388


In [ ]:
# 1. Hitung Q1 dan Q3 (seperti 'nilai batas bawah' dan 'nilai batas atas')
num_df = df.select_dtypes(include='number')
Q1 = num_df.quantile(0.25)
Q3 = num_df.quantile(0.75)
IQR = Q3 -  Q1

# Buat batas wajar
lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

# Ambil baris yang masih di dalam batas wajar
df_no_outliers = df[
    (num_df >= lower_limit) & (num_df <= upper_limit)
].all(axis=1) # SUPAYA PYTHON MENGECEK SEMUA BARIS YANG TERPENUHI KRITERIANYA
# JADI BUKAN YANG HANYA MEMENUHI SATU KRITERIA SAJA.

df_clean = df[df_no_outliers]
df_clean.head()

,Country,Region,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption),Generosity,Dystopia Residual
0,Finland,Western Europe,1,7.8210,1.891628,1.258108,0.775206,0.735590,0.533658,0.108733,2.518052
1,Denmark,Western Europe,2,7.6362,1.952595,1.242681,0.776644,0.718918,0.532079,0.187626,2.225632
2,Iceland,Western Europe,3,7.5575,1.935726,1.319914,0.802622,0.718194,0.191204,0.269616,2.320185
3,Switzerland,Western Europe,4,7.5116,2.025970,1.226074,0.822048,0.676947,0.461004,0.146822,2.152746
4,Netherlands,Western Europe,5,7.4149,1.944578,1.205848,0.786738,0.650682,0.419083,0.271076,2.136937


# CONCLUSION FOR DATA WITH OUTLIERS:

1. df_clean = dataset without outliers.
2. df = original dataset (unchanged).

# BEST PRACTICE WITHIN THE INDUSTRY

Tipe Analisis	Gunakan Data :
1. EDA dasar :	df asli
2. Deteksi missing value :	df asli
3. Outlier detection	: df asli
4. Model statistik (korelasi, regresi, clustering) :	df_clean
5. Uji kategorikal (Chi-Square)	: df_clean atau df asli

# Checking Section For Statistical Analysis

In [ ]:
number_df = df_clean.select_dtypes(include='number')
corr_for_pearson = number_df.corr(method='pearson')

In [ ]:
# PLOT FOR VISUALIZE
fig = px.imshow( # UNTUK MENAMPILKAN MATRIX KORELASI DARI TABEL ANGKA 2 DIMENSI.
    corr_for_pearson,
    text_auto=True,     # text_auto=False → kosong, text_auto=True → ada angka
    aspect='auto',  # plotly menyesuaikan ukuran horizontal dan vertikal
                    #  supaya heatmap tidak tertekan atau terjepit
    color_continuous_scale='Portland', # WARNA GRADASI KARENA HEATMAP PERLU WARNA GRADASI.
    title= 'Pearson Correlation For Happines Correlation'
)
fig.show() # HASILNYA YANG 1 SETERUNSYA SECARA DIAGONAL ITU KORELASI DENGAN DIRINYA SENDIRI.
# JADI TIDAK BOLEH DIPAKAI KARENA TIDAK ADA MAKNANYA UNTUK ARTI DARI KORELASI.
# JANGAN PAKAI YANG MAKNANYA HAMPIR MIRIP MAKNANYA SEPERTI HAPPINES_SCORE DENGAN HAPPINES_RANK
# KARENA Ranking kecil = score besar, Ranking besar = score kecil.
# DYSTOPIA / RESIDUAL =  sisa yang belum terjelaskan DALAM KAITANNYA DENGAN HAPPINES SCORE.

# INTERPRETATION :
1. “A country’s happiness level is more influenced by basic human factors: economic stability, health, and social support, than by moral factors like generosity.”

2. “Personal freedom and trust in government also contribute, but not as strongly as economic factors and health.”

3. “The happiness ranking is inversely proportional to the happiness score, thus indicating structural consistency in the data.”

In [ ]:
# SCATTER PLOT FOR HAPPINES VS ECONOMY

fig = px.scatter(
    df_clean,
    x='Economy (GDP per Capita)',
    y='Happiness Score',
    trendline='ols', # trendline="ols" = suruh Plotly menggambar garis prediksi
                    # otomatis menggunakan regresi linear sederhana.
                    # OLS = Ordinary Least Squares.
    title='Scatter : ECONOMY VS HAPPINES',
    labels={"Economy (GDP per Capita)":"Factor Economy","Happiness Score":"Happiness Factor"}
)
fig.show()

In [ ]:
# SCATTER PLOT HAPPINES SCORE VS FAMILY (SOCIAL SUPPORT)
fig = px.scatter(
    df_clean,
    x='Family',
    y='Happiness Score',
    trendline='ols',
    labels={'Family':'Family (SOcial Support)','Happiness Score':'Happiness Factor'},
    title='FAMILY VS HAPPINESS SCORE'
)
fig.show()

In [ ]:
# SCATTER PLOT HAPPINESS SCORE VS HEALTH (LIFE EXPENTANCY)
fig = px.scatter(
    df_clean,
    x='Health (Life Expectancy)',
    y='Happiness Score',
    trendline='ols',
    labels={'Health (Life Expectancy)':'Health (Life Expectancy)','Happiness Score':'Happiness Factor'},
    title='Health (Life Expectancy) VS Happiness Score'
)
fig.show()

In [ ]:
# SCATTER PLOT FREEDOM VS HAPPINESS SCORE
fig = px.scatter(
    df_clean,
    x='Freedom',
    y='Happiness Score',
    labels={'Freedom':'Freedom Factor','Happiness Score':'Happiness Factor'},
    title='Freedom VS Happiness Score',
    trendline='ols'
)
fig.show()

In [ ]:
fig = px.scatter(
      data_frame=df_clean,
      x='Generosity',
      y='Happiness Score',
      labels={'Generosity':'Generosity Factor','Happiness Score':'Happiness Factor'},
      title='Generosity VS Happiness Factor',
      trendline='ols'
)
fig.show()

In [ ]:
fig = px.scatter(
      data_frame=df_clean,
      x='Trust (Government Corruption)',
      y='Happiness Score',
      labels={'Trust (Government Corruption)':'Trust Factor','Happiness Score':'Happiness Factor'},
      title='Trust Factor On Government VS Happiness Factor',
      trendline='ols'
)
fig.show()

#  Graph Meaning :
Interpretation of the OLS trendline:

INTERPRETATION (Scatter Plot: Family vs Happiness Score)

1. There is a strong positive relationship between Family (Social Support) and Happiness Score. This means that the higher a country's social support, the higher its happiness level.

2. The slope = 3.03
This means: If the Social Support value increases by 1 unit, the Happiness Score tends to increase by about 3 points.

3. This is a very large effect, almost 3x stronger than the economic effect I found earlier (around 1.97).

4. R² = 0.604
This means: 60.4% of the variation in Happiness Score can be explained by Social Support alone. This is the largest contribution compared to economics, health, freedom, generosity, trust, etc.

Human meaning :
Social support is the most basic human need—feeling safe, having someone to help during a crisis, and it turns out to be a stronger driver of happiness rather than money, freedom, or health.

Conclusion:
Social Support is the strongest individual predictor of happiness in this dataset according with statistical method.


# CREATE LINIEAR REGRESSION MULTIPLE FOR ANALYZE HAPPINESS FACTOR

In [ ]:
# IMPORT STAT MODELS
import statsmodels.api as sm

# CREATE MULTIPE LINIEAR REGRESSION
x = df_clean[['Economy (GDP per Capita)', 'Family', 'Health (Life Expectancy)',
              'Freedom', 'Trust (Government Corruption)', 'Generosity']]
x = sm.add_constant(x) # ADD CONSTANTS FOR X
y = df_clean['Happiness Score']

model_liniear_Regression = sm.OLS(y,x).fit(cov_type='HC3')
print(model_liniear_Regression.summary())

                            OLS Regression Results                            
Dep. Variable:        Happiness Score   R-squared:                       0.773
Model:                            OLS   Adj. R-squared:                  0.763
Method:                 Least Squares   F-statistic:                     71.42
Date:                Fri, 28 Nov 2025   Prob (F-statistic):           9.47e-40
Time:                        17:13:00   Log-Likelihood:                -109.84
No. Observations:                 144   AIC:                             233.7
Df Residuals:                     137   BIC:                             254.5
Df Model:                           6                                         
Covariance Type:                  HC3                                         
                                    coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
const         

# Interpretation For P-Value Variable For Liniear Regression :
p < 0.05 → the variable has a significant effect

p ≥ 0.05 → the effect is weak/insignificant

# INTERPRETATION FOR LINIEAR REGRESSION (MULTIPLE) :
A. Model Strength (Goodness of Fit)

1. R-squared = 0.773
→ This means that 77.3% of the variation in happiness can be explained by these six variables.
This is a very good value for a social model.

2. Adjusted R-squared = 0.763
→ Adjusting for the number of variables.
The value is still high → the model is stable, not overfitting.

3. F-statistic = 77.66, p-value = 1.21e-41
→ Overall:
The model is significant.
This means that the combination of the six variables together effectively predicts the Happiness Score.

B. Interpretation of Factors
✔ Economy (coefficient 0.55, p=0.009)

1. If the economy increases by 1 unit, happiness increases by 0.55 points.
→ significant

2. ✔ Family (coefficient 1.40, p=0.000)
If social support increases by 1 unit, happiness increases by 1.40 points.
→ strongest and most significant factor
→ consistent with the previous scatter plot.

3. ✔ Health (coefficient 1.27, p=0.005)
Health has a strong influence.
✔ Freedom (coefficient 1.60, p=0.000)
Freedom contributes significantly to happiness.

4. ❗ Trust (coefficient 0.73, p=0.071)
Not significant (p>0.05)
→ Trust in government tends to have an effect, but it is not statistically strong enough.

5. ❗ Generosity (coef 0.95, p=0.102)
Not significant
→ Giving/generosity does not strongly predict happiness after other variables are controlled.

C. Main Conclusions

1. The prediction model is very good (R² = 0.77).

2. Factors that most strongly influence happiness are Family (Social Support), Freedom, Health
Factors that are still important but not significant are Trust, Generosity.

IN CONCLUSION :
Happiness is more influenced by social relationships, health, and personal freedom, than trust in government or generosity.

# CHECKING FOR Multicollinearity Check (VIF)

In [ ]:
# CHECKING FOR Multicollinearity Check (VIF)
from statsmodels.stats.outliers_influence import variance_inflation_factor # UNTUK CEK VIF
vif_df = pd.DataFrame()
vif_df['Variable'] = x.columns # X isinya yang tadi dipakai untuk Add Constant dan ADD COLUMNS.
vif_df['Vif_df'] = [variance_inflation_factor(x.values, column_index) for column_index in range(x.shape[1])]
print(vif_df)


                        Variable     Vif_df
0                          const  22.013431
1       Economy (GDP per Capita)   3.961923
2                         Family   2.399709
3       Health (Life Expectancy)   3.142634
4                        Freedom   1.561597
5  Trust (Government Corruption)   1.334940
6                     Generosity   1.146390


In [ ]:
# ######################### PENJELASAN CODE DI ATAS ################### #
# KARENA ADA FOR IN DI SITU MAKA HASILNYA MENJADI 7 KALI PERINTAH / LOOPING SEHINGGA
# VIF AKAN MENGUJI SETIAP SEMUA COLUMN_INDEX YANG ADA, JADI BUKAN HANYA 1 SAJA YAITU YANG
# URUTAN COLUMN_INDEX KE-7 KARENA KALAU PADA column_index diganti langsung 7 maka berarti
# HANYA MEMBACA DAN MEMPERHITUNGKAN URUTAN YANG KETUJUH.

# [ ] = Python list, bagian dari cara Python menyatakan “ini kumpulan data yang bisa di-loop”
# X.shape[0] → jumlah baris
# X.shape[1] → jumlah kolom
# MISALNYA = X.shape = (150, 5) .shape memberitahu “berapa banyak isi tabel ini”.
# (seperti melihat ukuran kertas: panjang × lebar)
# x.shape[0] → ambil jumlah baris
# x.shape[1] → ambil jumlah kolom
# TAPI:
# x.shape[1] bukan berarti “ambil semua kolom”.
# Itu hanya mengembalikan angka berapa jumlah kolom.
# Misal kolom ada 7 → hasilnya cuma angka 7, bukan kolomnya.

# range(n)?
# range(n) artinya:
# “buat angka dari 0 sampai n-1”.
# Kalau n = 5 → range(5) menghasilkan:
# 0, 1, 2, 3, 4
# Yaitu indeks kolom.

# FUNGSI VALUES PADA CODE TERSEBUT :
# .values mengubah DataFrame menjadi array tanpa nama → makanya pakai index numerik

################ LOGIC COLUMN_INDEX #######################
# x.values mengubah DataFrame menjadi array murni (tanpa nama kolom).
# Contoh:
# DataFrame:
# GDP	Family	Health
# ...	...	...

# Menjadi array:
# [[0.9, 1.2, 0.8],
# [0.8, 1.0, 0.7],
#  ...]


# Ketika sudah dalam bentuk array:
# ➡️ kolom tidak punya nama lagi
# ➡️ kolom hanya punya index:

# kolom 0 = GDP
# kolom 1 = Family
# kolom 2 = Health
# Itulah mengapa VIF memakai index, bukan nama.

# 🔥 3. Kenapa pakai range(x.shape[1])?

# x.shape[1] = jumlah kolom.
# Misal:
# x.shape = (144, 7)
# Artinya:
# 144 baris
# 7 kolom (const + 6 predictor)

# Maka:
# range(x.shape[1])
# → range(7)
# → [0, 1, 2, 3, 4, 5, 6]


# Kode ini artinya:
# “Loop dari kolom pertama sampai kolom terakhir.”
# Kenapa harus begitu?
# Karena VIF harus dihitung untuk setiap kolom, satu per satu.

# INTERPRETATION FOR MULTICOLINEARITY :
1. 🟦  No Multicollinearity Issues in the Model
All variables have a VIF < 5, indicating that:
The variables in the model do not replicate each other
There are no extreme overlapping variables
The regression model is reliable because the predictors operate independently. This means the model is stable and each variable provides new information, not simply repeating another variable.

2. 🟩 Explanation per Variable
GDP per Capita – VIF = 3.96
There is a slight correlation with other variables such as Health or Family. But still safe (below 5)
This means: GDP makes a unique contribution to happiness without causing model disruption.

3. Family (Social Support) – VIF = 2.40
The relationship between variables is moderate
This means: Social Support is a stand-alone predictor and does not overlap with GDP or Health.

4. Health – VIF = 3.14
Partially related to GDP (richer countries tend to have better health). But still safe Doesn't disrupt the model.

5. Freedom, Trust, Generosity – VIF 1.1–1.5
Almost no relationship between variables
These are the most “pure” variables.
They provide additional information that is largely independent.

"This model is very sound. There's no disturbing multicollinearity." All variables can be used together in linear regression without any problems. There's no need to drop any variables.

# CHECKING HETEROSKEDASTIK

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan

bp_test = het_breuschpagan(model_liniear_Regression.resid, model_liniear_Regression.model.exog)

print(f"Breuschpagan Pagan Stat : {bp_test[0]}")
print(f"Breuschpagan Pagan P-Value : {bp_test[1]}")

Breuschpagan Pagan Stat : 22.909174804125634
Breuschpagan Pagan P-Value : 0.0008274759829285851


#INTERPRETATION HETEROSCEDASTICITY:
a. If the p-value is < 0.05, there is heteroscedasticity (a problem).

b. If the p-value is ≥ 0.05, there is no heteroscedasticity (safe).

Official Interpretation for Presentation
Breusch-Pagan Test Results

BP statistic = 22.91
p-value = 0.00083 (<0.05)

1. Interpretation, Based on the Breusch-Pagan test, there are indications of heteroscedasticity in the model.This means that the error variance is not constant, so the standard error of the ordinary OLS model can be biased.

2. To ensure the model remains valid, I used heteroscedasticity-robust standard errors (HC3).

3.  After using HC3
Regression coefficients remain unchanged. R-squared remains the same. The overall F-statistic remains significant. Standard errors are corrected for more accurate estimates

4. Conclusion for the Model
Despite the presence of heteroscedasticity, the regression model remains valid because it has been corrected using robust standard errors (HC3).

Therefore, all interpretations of the coefficients, p-values, and significance of the model remain valid.

In [ ]:
# LETS VISUALIZE IT
residuals = model_liniear_Regression.resid # SISA / KESALAHAN MODELNYA.
fittedvalues =  model_liniear_Regression.fittedvalues # PREDIKSI DARI MODEL.

# MAKE IT TO DATAFRAME :
df_residual = pd.DataFrame({
      'Residual': residuals,
      'Fitted Values':fittedvalues})
# VISUALIZE IT
# LOWESS is like a “stabilizer” or “highlight marker” that makes it easier
# for the human eye to see patterns among the messy dots.

fig =  px.scatter(
    data_frame=df_residual,
    x='Fitted Values',
    y='Residual',
    trendline = 'lowess',
    labels = {'Fitted Values':'Fitted Values','Residual':'Residual'},
    title = 'HETEROSCEDASITY CHECK'
)
# ADD HORIZONTAL LINE FOR IT (it cannot be a straight line
#  because this is not a prediction) / Because the residuals do not have a
# linear relationship with the fitted values.

fig.add_hline(y=0, line_dash ='dash', line_width = 2)

fig.update_layout(
    width = 800,
    height = 600
)
fig.show()

# INTERPRETASION FOR HETEROSCEDASITY SCATTER PLOT :
"The plot of residuals vs. fitted values ​​shows a clear pattern. The LOWESS line is not flat around zero, but forms a curved pattern like an 'inverted mountain'. This indicates that the residual variance is not constant. In other words, the model suffers from heteroscedasticity, because the error is larger at some levels of prediction than at others. This pattern indicates that the model is not yet fully stable and its standard estimates may be biased."

# Finalization Part to Turn Prediction Model into DataFrame

In [ ]:
from sklearn.linear_model import LinearRegression

# GROUP THE DATA FOR X AND Y VARIABLES
X = df_clean[['Economy (GDP per Capita)',
    'Family',
    'Health (Life Expectancy)',
    'Freedom',
    'Trust (Government Corruption)',
    'Generosity']]

Y = df_clean['Happiness Score']

# CREATE ASSIGN FOR MODEL FIT
model_Sklearn = LinearRegression() # FUNCTION
model_Sklearn.fit(X,Y)

# PROTECT DATA SCOURCE CLEAN WITH EDA BEFORE WITH copy()
df_model =  df_clean.copy()

# PREDICT X FOR Y
df_model['Happiness Predict'] = model_Sklearn.predict(X)


In [ ]:
df_model.head(5)

,Country,Region,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption),Generosity,Dystopia Residual,Happiness Predict
0,Finland,Western Europe,1,7.8210,1.891628,1.258108,0.775206,0.735590,0.533658,0.108733,2.518052,7.147346
1,Denmark,Western Europe,2,7.6362,1.952595,1.242681,0.776644,0.718918,0.532079,0.187626,2.225632,7.208262
2,Iceland,Western Europe,3,7.5575,1.935726,1.319914,0.802622,0.718194,0.191204,0.269616,2.320185,7.168038
3,Switzerland,Western Europe,4,7.5116,2.025970,1.226074,0.822048,0.676947,0.461004,0.146822,2.152746,7.125033
4,Netherlands,Western Europe,5,7.4149,1.944578,1.205848,0.786738,0.650682,0.419083,0.271076,2.136937,7.052035


In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

y_true = df_model['Happiness Score']
y_pred = df_model['Happiness Predict']   # GANTI sesuai nama kolom prediksi Anda

print(f"RMSE: {root_mean_squared_error(y_true, y_pred)}")
print(f"R2 Score: {r2_score(y_true, y_pred)}")
print(f"MAPE: {mean_absolute_percentage_error(y_true, y_pred)}")

RMSE: 0.5188352382570506
R2 Score: 0.7727892232556359
MAPE: 0.08325640616273938


# INTERPRETATION FOR ANALZE BASED ON THE MODEL
1. The constructed multiple linear regression model explained 77% of the variation in Happiness Score values ​​based on the variables of economics, family, health, freedom, trust in government, and generosity.

2. The model's prediction error rate was relatively low, with an RMSE of 0.52, meaning the average difference between actual and predicted values ​​was only about half a point on the happiness scale.

3. Furthermore, the MAPE value of 8.3% indicates that the model's relative error is quite small, indicating that the model performs well and is suitable for use in predictive happiness analysis on the 2022 dataset.

In [ ]:
model_Sklearn.coef_

array([0.55141847, 1.40461112, 1.27648172, 1.60759958, 0.73176048,
       0.95179088])

In [ ]:
model_Sklearn.intercept_

np.float64(1.6710423467177975)

# MAKING CHI SQUARE TESTING

In [ ]:
# We grouped countries into three happiness levels based on their score positions
#  across the entire dataset, rather than guessing.
# This method is called quantile-based categorization.

# 1. BUAT DATA DF KHUSUS UNTUK CHI DAN BUKAN MENGANGGU DF_CLEAN
df_chi =  df_clean.copy()

# 2. BUAT KATEGORI KEBAHAGIAAN BERDASARKAN SCORE-NYA.
df_chi['Happiness Category'] = pd.qcut(df_chi['Happiness Score'], q=3, # MAKSUDNYA quantile
                                      labels=['Low','Medium','High']) # PAKAI LIST KARENA HASILNYA URUTAN.
                              # kalau labels = {pakai ini} maka Itu adalah SET, BUKAN UNTUK URUTAN,
                              #DAN BISA MENYEBABKAN KATEGORI ACAK, JUSTRU BUKAN BERDASARKAN TINGKAT SCORENYA.
df_chi.head(5)

,Country,Region,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption),Generosity,Dystopia Residual,Happiness Category
0,Finland,Western Europe,1,7.8210,1.891628,1.258108,0.775206,0.735590,0.533658,0.108733,2.518052,High
1,Denmark,Western Europe,2,7.6362,1.952595,1.242681,0.776644,0.718918,0.532079,0.187626,2.225632,High
2,Iceland,Western Europe,3,7.5575,1.935726,1.319914,0.802622,0.718194,0.191204,0.269616,2.320185,High
3,Switzerland,Western Europe,4,7.5116,2.025970,1.226074,0.822048,0.676947,0.461004,0.146822,2.152746,High
4,Netherlands,Western Europe,5,7.4149,1.944578,1.205848,0.786738,0.650682,0.419083,0.271076,2.136937,High


In [ ]:
# Kalau diterjemahkan ke bahasa manusia:
# "Tolong hitung berapa kali setiap kategori kebahagiaan muncul di setiap wilayah."

# Bukan random.
# Bukan sekadar teknis.
# Tapi ini setara dengan:

# 📊 Excel → Pivot Table / TABLE SILANG
# Rows: Region
# Columns: Happiness Category
# Values: Count

# Jadi Anda sebenarnya sedang:
# ✅ Menyusun data mentah menjadi bentuk yang bisa dianalisis
# ✅ Mengubah angka acak menjadi pola bermakna

In [ ]:
contingency_table = pd.crosstab(
    df_chi['Happiness Category'], df_chi['Region']
)
contingency_table

# ✅ Gunakan reset_index jika:

# Anda ingin:
# Menyimpan ke CSV
# Menampilkan di laporan
# Mengolah ulang sebagai dataframe biasa
# Visualisasi yang butuh kolom kategori

# ❌ Jangan reset_index jika:
# Tabel tersebut langsung akan dipakai untuk:
# Chi-Square
# Perhitungan statistik numerik
# Kalau sudah di-reset, maka kolom teks ikut masuk
#  → bisa memicu error atau hasil salah.

Region,Australia and New Zealand,Central and Eastern Europe,Eastern Asia,Latin America and Caribbean,Middle East and Northern Africa,North America,Southeastern Asia,Southern Asia,Sub-Saharan Africa,Western Europe
Happiness Category,,,,,,,,,,
Low,0,1,0,1,10,0,2,4,30,0
Medium,0,13,5,9,5,0,6,3,5,2
High,2,11,1,9,4,2,1,0,0,18


In [ ]:
from scipy.stats import chi2_contingency
alpha = 0.05
chi2, p_value, dof, expected = chi2_contingency(contingency_table)
print("Alpha Value :",alpha)
print("Chi-square:", chi2)
print("p-value:", p_value)
print("Degrees of freedom:", dof)
# 1.69e-16 artinya:
# 0.000000000000000169
# Why does it feel backwards?
# Because linguistically, we expect:
# "If it's small, it means it's true."
# But in statistics, it's not that small means it's true, but rather:
# "Small means this event is highly unlikely to have occurred by chance."

Alpha Value : 0.05
Chi-square: 116.78666666666668
p-value: 1.69108644453719e-16
Degrees of freedom: 18


In [ ]:
print(f"EXPECTED : \n\n {expected}")

EXPECTED : 

 [[ 0.66666667  8.33333333  2.          6.33333333  6.33333333  0.66666667
   3.          2.33333333 11.66666667  6.66666667]
 [ 0.66666667  8.33333333  2.          6.33333333  6.33333333  0.66666667
   3.          2.33333333 11.66666667  6.66666667]
 [ 0.66666667  8.33333333  2.          6.33333333  6.33333333  0.66666667
   3.          2.33333333 11.66666667  6.66666667]]


H0 = "Region is not related to happiness level"
HA = "Region is related to happiness level"

In [ ]:
# UJI TESTING :

In [ ]:
if p_value <= alpha:
    print("Dependent (Reject H0)")
    print("Region is related to the level of happiness")
else:
    print("Independent (Fail to Reject H0)")
    print("There is no proven relationship between Region and happiness levels.")

Dependent (Reject H0)
Region is related to the level of happiness


# CHI SQUARE INTERPRETATION :
Based on the Chi-Square test, the p-value obtained was 1.69 × 10⁻¹⁶, which is much smaller than α = 0.05. This indicates a significant relationship between Region and happiness level categories. Thus, the null hypothesis is rejected, meaning the distribution of happiness levels does not occur randomly across regions, but is influenced by regional factors.

# LAST ONE DONT FORGET STATISTICAL DESCRIPTIVE

In [ ]:
# PIVOT WIDE :
# ✅ Each factor remains independent
# ✅ Nothing is mixed
# ✅ Everything is clean and aggregated per region

pivot_factors = df_clean.pivot_table(
        values = ['Economy (GDP per Capita)',
        'Family',
        'Health (Life Expectancy)',
        'Freedom',
        'Trust (Government Corruption)',
        'Generosity'],
        index='Region',
        aggfunc='mean'
)

# UBAH JADI LONG PLOTING :
pivot_long = pivot_factors.reset_index().melt(
      id_vars = 'Region',
      var_name = 'factor',
      value_name = 'Average Values'
)

# SORT DULU
region_order = (pivot_long.groupby('Region')['Average Values']
                .mean().sort_values(ascending=False).index
                )
# Ini bukan data grafik.
# Ini adalah:
# Daftar urutan Region dari nilai rata-rata terbesar → terkecil
# region_order

In [ ]:
# VISUALIZE IT
fig = px.bar(
    data_frame=pivot_long,
    x='Region',
    y='Average Values',
    title = 'Average Happiness Factors by Region',
    color = 'factor',
    labels = {'Region':'REGION','Average values':'AVERAGE VALUES'},
    barmode='group',
    category_orders={"Region":region_order}
)
fig.show()

In [ ]:
# RINGKASAN CODE KENAPA PAKAI LONG TABLE ??????
# Intinya dulu (jawaban singkat tapi tepat)
# 1. ✅ Ya, pemahamanmu BENAR dengan satu penyesuaian penting:
# Wide table ≈ seperti tabel Excel yang melebar ke samping (banyak kolom variabel)
# Long table ≈ seperti tabel Excel yang memanjang ke bawah (variabel dijadikan baris)

# Tapi bukan sekadar soal arah visual, melainkan:
# 👉 soal cara data direpresentasikan untuk dianalisis & divisualisasikan

# 2. Contoh konkret agar tidak abstrak
# Bayangkan kamu punya data begini di Excel:

# ✅ WIDE (melebar)
# Region      Economy   Family   Health
# Europe      1.35      1.28     0.88
# Asia        0.98      1.10     0.75


# Ciri logisnya:
# Setiap variabel punya kolom sendiri
# Cocok untuk:

# Perhitungan agregat
# Korelasi
# Heatmap
# Regresi

# Ini adalah bentuk "analitis struktural".
# ✅ LONG (memanjang)
# Region   Factor    Value
# Europe   Economy   1.35
# Europe   Family    1.28
# Europe   Health    0.88
# Asia     Economy   0.98
# Asia     Family    1.10


# Ciri logisnya:
# Semua faktor disatukan dalam satu kolom
# Variabel menjadi kategori

# Cocok untuk:
# Bar chart
# Grouped chart
# Facet plot
# Ini adalah bentuk "visual komunikatif".

# 3. Kenapa wide cocok untuk heatmap?
# Karena heatmap memerlukan struktur seperti matriks:
# Region	Economy	Family	Health
# Europe	1.35	1.28	0.88
# Heatmap membaca:
# Baris = entitas (Region)
# Kolom = variabel
# #Nilai = warna

# Jadi:
# [Europe, Economy] = 1.35 → warna lebih gelap
# Itu hanya bisa dilakukan jika bentuk datanya WIDE.

# INSIGHT AVERAGE HAPPINESS FACTOR BASED ON REGION :
The visualization shows that the Economy (GDP per capita) factor is the most dominant determinant in almost all regions, particularly Australia & New Zealand, North America, and Western Europe. This demonstrates that economic well-being remains the primary foundation of measurable happiness. This is followed by Family and Health factors, which are relatively stable in developed regions, indicating social support and a more secure quality of life.

Conversely, regions such as Sub-Saharan Africa and Southern Asia show low scores in almost all factors, particularly Economy, Trust, and Health, indicating structural limitations and weak life support systems. Interestingly, the Generosity and Freedom factors are not always high in economically strong regions, indicating that happiness is not simply a matter of material things, but also the quality of relationships and social freedom. This reinforces the notion that happiness in developed regions tends to be "stable but fragile" because it is highly dependent on the stability of economic and institutional systems.

# RECOMMENDATION AVERAGE HAPPINESS FACTOR BASED ON REGION :
Referring to Erich Fromm's thoughts in *To Have or To Be?*, this data demonstrates the need for a paradigm shift in development from a focus on "having" to "being." If happiness continues to be reduced solely to economic figures, society risks becoming trapped in an existentially fragile illusion of well-being.

Therefore, the recommendation is to encourage social policies and approaches that not only increase income but also strengthen the meaning of life, social solidarity, and the quality of human relationships. This way, happiness will not only be statistically high but also more authentic, deeply rooted, and sustainable from a human perspective.

# DESCRIPTIVE STATISTICS TO ENSURE THE TRUTH OF REALITY CAPTURED BY CHI SQUARE ANALYSIS

In [ ]:
pivot_happiness_region = df_clean.pivot_table(
        values='Happiness Score',
        index = 'Region',
        aggfunc='mean'
).sort_values(by='Happiness Score', ascending=False)

pivot_happiness_region

,Happiness Score
Region,
Australia and New Zealand,7.180950
North America,7.000950
Western Europe,6.967100
Latin America and Caribbean,5.950353
Central and Eastern Europe,5.937268
Eastern Asia,5.876217
Southeastern Asia,5.431744
Middle East and Northern Africa,5.185868
Sub-Saharan Africa,4.479706


In [ ]:
fig = px.pie(
      data_frame=pivot_happiness_region.reset_index(),
      names='Region',
      values='Happiness Score',
      title = 'Proportion of Average Happiness Score by Region'
)
# title: teks judul legend (label grup warna).
# orientation='h': tata letak horizontal (legend di satu baris) — alternatif 'v' (vertikal).
# x dan xanchor bekerja sama seperti pada title: mengatur posisi legend.

fig.update_layout(
    title = dict(text="Proportion of Average Happiness Score by Region", x=0.5, xanchor='center'),
    font = dict(family='Arial', size=14),
    legend = dict(title='Region', orientation='h', x=0.5, xanchor='center'))
# Kalimat pegangan:
# "Kalau mau ubah tampilan, pikirkan elemen apa, lalu isi konfigurasinya sebagai dict."

fig.show()

# Kenapa pakai dict (alasan logis) SINGKATNYA KARENA TERSIMPAN LAYOUT INTERNALNYA DENGAN JSON
# YANG MEMAKAI STRUKTUR DICT UNTUK MENYIMPAN PROPERTI-PROPERTINYA.
# Banyak properti punya sub-properti → butuh struktur bertingkat.
# Contoh: title → { text, x, y, xanchor, yanchor, font }.
# dict merepresentasikan setting group dengan rapi dan eksplisit.
# update_layout menggabungkan (merge) dict yang kamu berikan ke struktur
# layout internal — jadi hanya bagian yang kamu tentukan yang berubah.

# Mengapa editor (IDE) tidak selalu menampilkan semua key?

# update_layout menerima **kwargs (arbitrary keyword args); Plotly menyokong ratusan
# properti → editor tak bisa menampilkan semuanya sebagai signature.

# Solusinya: gunakan dokumentasi Plotly atau inspeksi fig.layout ketika ragu.
# (Tapi kamu tidak perlu menghafal semuanya — cukup hafal elemen populer:
#  title, font, legend, margin, xaxis, yaxis, template.)

In [ ]:
# PENGATURAN SINGKAT UNTUK MELIHAT LAYOUT ADA APA SAJA :
#  Cara inspeksi (lihat apa yang sudah ada)

# Kalau ragu properti apa yang ada, lihat layout aktual:
# print(fig.layout)             # objek layout
# print(fig.to_plotly_json())   # lihat struktur JSON penuh


# Atau:

# fig.layout.title  # lihat dict title
# fig.layout.margin
# Ini berguna untuk melihat nama-nama properti dan nested structure.

#  Dapatkan daftar property (lebih teknis tapi berguna)
# import plotly.graph_objects as go
# print(go.Layout().to_plotly_json().keys())  # list top-level properti layout

# → Ini menampilkan top-level names yang valid (title, font, legend, margin, xaxis, yaxis, dsb).

# Cara lain mengakses / ubah (agar kamu paham struktur internal)

# Alternatif akses yang lebih eksplisit (berguna saat debugging):

# Set cara "objek" (dot access)
# fig.layout.title.text = "Proportion of Average Happiness Score by Region"
# fig.layout.title.x = 0.5
# fig.layout.title.xanchor = 'center'
# fig.layout.font.size = 14
# fig.layout.legend.title.text = 'Region'


# Atau lihat strukturnya:

# print(fig.layout)            # melihat objek layout
# print(fig.to_plotly_json())  # lihat struktur JSON penuh


# Humanis: kamu bisa membuka wadah fig.layout dan melihat semua "switch" yang bisa diatur.

In [ ]:
# Singkatnya dulu (intinya)

# dict = dictionary di Python → struktur data
# pasangan key : value (mirip kolom/field di Excel atau properti pada objek).
# fig.update_layout(title = dict(...)) artinya: “isi properti title pada
# layout dengan sekumpulan pengaturan (key: value)”.
# dict dipakai karena banyak properti layout itu bersarang (nested) —
# mis. title punya sub-properti text, x, xanchor, dll.
# Visualnya: dict adalah konfigurasi/setting object untuk sebuah elemen grafik.

# Bayangkan kamu memesan print poster ke percetakan.
# fig = poster yang sedang kamu desain.
# update_layout = formulir desain yang kamu isi.
# title = dict(...) = bagian formulir “Judul poster” — di sana kamu isi:
# text (isi judul),
# x (posisi horizontal),
# xanchor (bagian mana dari teks yang dipakai sebagai titik referensi).
# Jadi dict itu seperti bagian formulir khusus untuk fitur tertentu.



# INTERPRETATION BASED ON THE PIE CHART I CREATED :
The graph shows that global happiness is not evenly distributed. Australia & New Zealand, North America, and Western Europe dominate the happiness score, indicating that economic stability, quality of public services, and social security play a significant role in shaping a consistent quality of life.

Conversely, Sub-Saharan Africa and Southern Asia have the lowest proportions, indicating structural limitations such as poor access to basic necessities and weak social resilience. Although developed regions excel, happiness there tends to be "stable but fragile" in the face of economic and social crises, as it is highly dependent on the sustainability of the systems that support it.

# RECOMMENDATION BASED ON THE PIE CHART I CREATED :
Referring to Erich Fromm, happiness should not be pursued solely through material improvements and statistical indicators, but through a shift from an orientation toward "having" to "being." This means that development should not only target economic growth but also strengthen the quality of human relationships, a sense of meaning, and a sense of life.

With this approach, regions with high scores can deepen the quality of their happiness beyond mere numerical stability, while regions with low scores can build a more humane and sustainable foundation for happiness.